# L3b: Data Validation and Provenance

A file that parses is not a file you can trust. In this lab we read fulfillment-center shift records from CSV and their metadata from JSON, inspect provenance before values, validate the data contract, diagnose a deliberately broken file, and verify byte-level integrity.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Read a data bundle as one artifact:__ Load tabular records together with the structured metadata that describes them, and treat the pair rather than the table alone as the thing you received.
> * __Interrogate provenance before values:__ Determine what a dataset is, who made it, and which claims it cannot support, before any number from it reaches a calculation.
> * __Turn a contract into executable checks:__ Express schema, ranges, uniqueness, and row counts as validation code, and read a failure report as a repair list rather than a single stop-on-first-error.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources. The class-meeting folder holds the records file, its metadata, a known-bad example, the checksum record, and the source code, so this notebook needs no network access.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the local setup file in the notebook's global scope. This local file delegates environment activation to the repository root and loads the L3b source module.

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

### The data

These are the same shift records built by hand in the Week 2 collections lecture — `shift_id`, `zone`, `orders_completed`, `labor_hours` — with one column added, `picking_error_fraction`, the share of picked orders later found to be wrong. The difference is that this time they arrive as a file from somewhere else, and nothing about a file obliges it to be correct.

Eight shifts, four per zone. The CSV holds the records; the JSON holds everything you need in order to decide whether to believe them: origin, purpose, units, expected row count, allowed zones, and the constraints each column must satisfy.

> __Parsing versus validation:__
>
> A parser decides whether bytes follow CSV or JSON syntax. It has no idea that `labor_hours` must be positive, that `picking_error_fraction` is a fraction and so cannot exceed one, that a zone must be one the facility actually operates, or that shift identifiers must be unique. Those are not syntax questions. They belong to the __data contract__, and the only way a contract gets enforced is if someone writes it down as code.

Let's name the four paths we will work with:

In [ ]:
data_root = joinpath(CHEME5800_L3B_ROOT, "data");
csv_path = joinpath(data_root, "fulfillment-shifts.csv");
metadata_path = joinpath(data_root, "fulfillment-metadata.json");
invalid_path = joinpath(data_root, "fulfillment-shifts-invalid-example.csv");

___

## Task 1: Inspect provenance before observations

It is tempting to open a data file and look at the numbers. Resist it for one cell. What the numbers _are_ determines what you are allowed to conclude from them, and that information lives in the metadata, not the table.

So before we print a single row, let's ask where this came from:

In [ ]:
bundle = L3bData.load_shift_bundle(csv_path, metadata_path);
provenance_record = (
    dataset_id = bundle.metadata["dataset_id"],
    provenance = bundle.metadata["provenance"],
    purpose = bundle.metadata["purpose"],
    units = bundle.metadata["units"],
)

The record answers the question plainly: instructor-generated synthetic values, created to practice a computational workflow. That is a licence to test code against them and to reason about file structure. It is not a licence to compare the two zones' real productivity, to estimate a staffing requirement, or to put a number from this file into a report.

Provenance is not paperwork. It is the boundary on what any downstream conclusion can legitimately say.

___

___

## Task 2: Validate the authored records

Now the contract. The loader parses both files and then checks, in one pass: that every required column is present; that identifiers are nonempty and unique; that each zone is one the facility operates; that order counts are whole and not negative; that labor hours are positive; that the error fraction lies in $[0,1]$; and that the row count matches what the metadata claims.

An empty error list is the only acceptable result. So what do we get?

In [ ]:
bundle.validation

No errors, so the authored files satisfy the declared contract. Now — and only now — we look at the table.

Note what passing did _not_ do: it did not make the data real. The provenance is unchanged. Validation establishes internal consistency, never truth.

In [ ]:
bundle.shifts

___

## Task 3: Diagnose a known-bad example

A validator nobody has watched fail is a validator nobody has tested. The second CSV is broken on purpose, and it is broken in six different ways at once:

> __What is wrong with it:__
>
> * `S01` appears twice, so the identifiers are not unique.
> * One row has a blank zone.
> * Another names zone `North`, which the facility does not operate.
> * One row reports $-4$ orders completed.
> * One row reports zero labor hours, which would make throughput infinite.
> * One row reports a picking error fraction of $1.07$, which is not a fraction.

We keep this file in the repository on purpose. A reproducible failure case is the only way to know the validator still works after someone edits it.

Does the report catch all six?

In [ ]:
invalid_shifts = CSV.read(invalid_path, DataFrame);
invalid_report = L3bData.validate_shift_records(invalid_shifts)

All six, in a single pass. That design choice matters: a validator that stops at the first problem forces the data owner into a repair-rerun-repair loop, one defect at a time. Returning the complete list lets them fix everything and revalidate once.

Notice also that the errors name the row and the column. "Invalid data" is not an actionable message; `row 2: labor_hours must be finite and positive` is.

___

___

## Task 4: Verify integrity, and the same contract in Python

Validation checks that the _values_ obey the contract. It says nothing about whether the file is the one that was authored. A truncated download, a stray edit, or a partially-written file can all still parse and still validate.

A cryptographic digest closes that gap. We compute the SHA-256 of the CSV and compare it against the digest recorded in [`data/README.md`](data/README.md). A match establishes that our bytes are the authored bytes — and nothing more. It does not make synthetic values true.

In [ ]:
csv_sha256 = L3bData.file_sha256(csv_path);
authored_sha256 = "8674d24b30b5a3bf9dca5be28fcb8242fe8591ccb3118b9f048a51c1c85005ae"; # recorded in data/README.md
(sha256 = csv_sha256, matches_authored_file = csv_sha256 == authored_sha256)

[`src/shift_data.py`](src/shift_data.py) expresses the same record and metadata checks using only the Python standard library. The syntax differs, the runtime types differ, the exception vocabulary differs — the contract does not. A data contract is a property of the data, not of the language you happen to check it with.

From the repository root:

```bash
python weeks/week-03/L3b/src/shift_data.py
python -m unittest discover -s weeks/week-03/L3b/src -p 'test_*.py'
```

Finally, the tests that pin everything this lab claimed:

In [ ]:
@testset "L3b data contract" begin
    @test bundle.validation.valid
    @test nrow(bundle.shifts) == bundle.metadata["expected_rows"]
    @test occursin("synthetic", lowercase(bundle.metadata["provenance"]))
    @test !invalid_report.valid
    @test any(error -> occursin("unique", error), invalid_report.errors)
    @test any(error -> occursin("zone", error), invalid_report.errors)
    @test any(error -> occursin("orders_completed", error), invalid_report.errors)
    @test any(error -> occursin("labor_hours", error), invalid_report.errors)
    @test any(error -> occursin("picking_error_fraction", error), invalid_report.errors)
    @test csv_sha256 == authored_sha256
end

___

## Summary
A data bundle is records plus the metadata that says what they are, and both halves have to be checked before anything downstream is entitled to use them.

> __Key Takeaways:__
>
> * **Provenance bounds the conclusions:** What a dataset is, and who made it, decides which claims it can support. Reading that before the values is what stops a synthetic teaching file from turning into a quoted operational result.
> * **A contract is executable domain knowledge:** Units, ranges, allowed categories, uniqueness, and row counts are assumptions someone made about the data, and writing them as validation code is what turns them from assumptions into checks.
> * **Failing well is a design decision:** Reporting every violation in one pass, each naming its row and column, turns a validator from an obstacle into a repair list.

The same contract, enforced in Julia and in Python, produced the same verdict. That is the point: the requirements belong to the data, and the language is an implementation detail.
___